In [3]:
import pandas as pd
from pathlib import Path

In [22]:
dataset = Path("/home/arnav-gupta/Projects/itops_agent/src/itops_agent/classifier/dataset/annotated_dataset.csv")

In [23]:
df = pd.read_csv(dataset)

print("Dataset loaded successfully!")
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

display(df.head())

Dataset loaded successfully!
Shape: (7343, 3)

Columns:
['type', 'ticket', 'complexity']


,type,ticket,complexity
0,Incident,Account Disruption\n\nDear Customer Support Te...,multi_hop
1,Request,"Feature Query\n\nDear Customer Support,\n\nI h...",single_hop
2,Incident,System Interruptions\n\nDear Customer Support ...,multi_hop
3,Incident,Connectivity Problems with Printer on MacBook ...,multi_hop
4,Problem,Immediate Help Needed: Technical Problem with ...,multi_hop


In [25]:
df = df[df["complexity"].str.strip().str.lower() == "single_hop"].copy()
df.shape

(2245, 3)

In [27]:
df.head()

,type,ticket,complexity
1,Request,"Feature Query\n\nDear Customer Support,\n\nI h...",single_hop
5,Request,Enhancing Multi-Unit Marketing Processes\n\nDe...,single_hop
6,Request,Query About Future Service Disruptions and Mai...,single_hop
7,Change,Proposal for Upgrades to Big Data Analytics Sy...,single_hop
8,Request,Guidelines for Incorporating Seagate Expansion...,single_hop


In [28]:
df = df.head(1000).copy()

In [29]:
df.shape

(1000, 3)

In [30]:
df["llm_response"] = ""
df["slm_response"] = ""
df["similarity_score"] = pd.NA
df.head()

,type,ticket,complexity,llm_response,slm_response,similarity_score
1,Request,"Feature Query\n\nDear Customer Support,\n\nI h...",single_hop,,,<NA>
5,Request,Enhancing Multi-Unit Marketing Processes\n\nDe...,single_hop,,,<NA>
6,Request,Query About Future Service Disruptions and Mai...,single_hop,,,<NA>
7,Change,Proposal for Upgrades to Big Data Analytics Sy...,single_hop,,,<NA>
8,Request,Guidelines for Incorporating Seagate Expansion...,single_hop,,,<NA>


In [ ]:
import requests
from getpass import getpass

OPENROUTER_API_KEY = getpass("Enter your OpenRouter API key: ")

OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"
LLM_MODEL = "nvidia/nemotron-3-ultra-550b-a55b:free"


def get_llm_response(question):
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json"
    }

    payload = {
        "model": LLM_MODEL,
        "messages": [
            {
                "role": "system",
                "content": (
                    "You are an ITOps assistant. Provide an accurate and direct "
                    "resolution in strictly under 200 characters. Do NOT include "
                    "greetings, conversational filler, markdown headings, or preamble. "
                    "Output only the concise solution text (strictly at most 200 characters)."
                )
            },
            {
                "role": "user",
                "content": (
                    f"Ticket: {question}\n\n"
                    "Constraint: Provide only the direct answer in strictly under 200 characters."
                )
            }
        ],
        "max_tokens": 70
    }

    try:
        response = requests.post(
            OPENROUTER_URL,
            headers=headers,
            json=payload,
            timeout=180
        )

        response.raise_for_status()

        result = response.json()
        answer = result["choices"][0]["message"].get("content")

        if not answer:
            return None

        # Enforce strict 200 characters limit
        cleaned_answer = answer.strip()
        return cleaned_answer[:200]

    except requests.RequestException as e:
        print(f"OpenRouter API error: {e}")
        return None

    except (KeyError, IndexError, ValueError) as e:
        print(f"Unexpected OpenRouter response: {e}")
        return None


In [ ]:
import requests

OLLAMA_URL = "http://127.0.0.1:11434/api/chat"
SLM_MODEL = "qwen2.5:3b"


def get_slm_response(question):
    payload = {
        "model": SLM_MODEL,
        "messages": [
            {
                "role": "system",
                "content": (
                    "You are an ITOps assistant. Provide an accurate and direct "
                    "resolution in strictly under 200 characters. Do NOT include "
                    "greetings, conversational filler, markdown headings, or preamble. "
                    "Output only the concise solution text (strictly at most 200 characters)."
                )
            },
            {
                "role": "user",
                "content": (
                    f"Ticket: {question}\n\n"
                    "Constraint: Provide only the direct answer in strictly under 200 characters."
                )
            }
        ],
        "stream": False,
        "options": {
            "num_predict": 70
        }
    }

    try:
        response = requests.post(
            OLLAMA_URL,
            json=payload,
            timeout=180
        )

        response.raise_for_status()

        result = response.json()
        answer = result["message"].get("content")

        if not answer:
            return None

        # Enforce strict 200 characters limit
        cleaned_answer = answer.strip()
        return cleaned_answer[:200]

    except requests.RequestException as e:
        print(f"Ollama API error: {e}")
        return None

    except (KeyError, ValueError) as e:
        print(f"Unexpected Ollama response: {e}")
        return None


In [41]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Load the embedding model once
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")


def calculate_similarity(llm_response, slm_response):
    if not isinstance(llm_response, str) or not llm_response.strip():
        return None

    if not isinstance(slm_response, str) or not slm_response.strip():
        return None

    embeddings = embedding_model.encode(
        [llm_response, slm_response],
        normalize_embeddings=True
    )

    score = cosine_similarity(
        embeddings[0].reshape(1, -1),
        embeddings[1].reshape(1, -1)
    )[0][0]

    return float(score)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1684.37it/s]


In [44]:
import time

BATCH_SIZE = 50
output_path = dataset.parent / "single_hop_1000.csv"

# Invalidate any legacy responses exceeding 200 characters
mask_llm_long = df["llm_response"].fillna("").astype(str).str.len() > 200
df.loc[mask_llm_long, "llm_response"] = ""

mask_slm_long = df["slm_response"].fillna("").astype(str).str.len() > 200
df.loc[mask_slm_long, "slm_response"] = ""

# Save the current dataset before processing
df.to_csv(output_path, index=False)

# Identify rows that still need processing
def is_blank(value):
    return pd.isna(value) or (
        isinstance(value, str) and not value.strip()
    )

pending_indices = [
    idx for idx in df.index
    if (
        is_blank(df.at[idx, "llm_response"])
        or is_blank(df.at[idx, "slm_response"])
        or pd.isna(df.at[idx, "similarity_score"])
    )
]

total_pending = len(pending_indices)

print(f"Rows to process: {total_pending}")

for position, idx in enumerate(pending_indices, start=1):
    question = df.at[idx, "ticket"]

    try:
        # Generate baseline LLM response (strictly <= 200 characters)
        if is_blank(df.at[idx, "llm_response"]):
            llm_res = get_llm_response(question)
            df.at[idx, "llm_response"] = (
                llm_res.strip()[:200] if llm_res else ""
            )

        # Generate SLM response (strictly <= 200 characters)
        if is_blank(df.at[idx, "slm_response"]):
            slm_res = get_slm_response(question)
            df.at[idx, "slm_response"] = (
                slm_res.strip()[:200] if slm_res else ""
            )

        # Calculate similarity when both responses are available
        llm_answer = df.at[idx, "llm_response"]
        slm_answer = df.at[idx, "slm_response"]

        if (
            not is_blank(llm_answer)
            and not is_blank(slm_answer)
        ):
            score = calculate_similarity(llm_answer, slm_answer)
            df.at[idx, "similarity_score"] = (
                score if score is not None else pd.NA
            )

    except Exception as e:
        print(f"Row {idx} failed: {e}")

    # Save after every 50 processed rows and at the end
    if position % BATCH_SIZE == 0 or position == total_pending:
        df.to_csv(output_path, index=False)
        print(
            f"Progress: {position}/{total_pending} processed. "
            f"Saved to {output_path}"
        )

print("Processing finished.")
print("Final dataset shape:", df.shape)


Rows to process: 1000


KeyboardInterrupt: 